## Import Libraries

In [14]:
import os

os.environ["HSA_OVERRIDE_GFX_VERSION"] = "10.3.0"

In [15]:
import numpy as np

import torch 
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

import matplotlib.pyplot as plt

## Prepare train dataset

In [16]:
# Transform to turn into tensor and normalize
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307),(0.3081,)) # mean y std
])

In [17]:
# Load the dataset and normalize it
train_dataset = datasets.MNIST(
    root='./data',
    train=True,
    download=False,
    transform=transform
)

## Prepare test dataset

In [18]:
transform_test = transforms.ToTensor()

In [19]:
# Load the test dataset
test_dataset = datasets.MNIST(
    root='./data',
    train=False,
    download=False,
    transform=transform_test
)

## Data Loaders

In [20]:
# DataLoaders
train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=32,
    shuffle=True,
    drop_last=True
)

test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=len(test_dataset[0][0][0])
)

## Feed Forward Network 

In [21]:
# All variables
%whos

Variable                 Type          Data/Info
------------------------------------------------
DataLoader               type          <class 'torch.utils.data.dataloader.DataLoader'>
F                        module        <module 'torch.nn.functio<...>/torch/nn/functional.py'>
X                        Tensor        tensor([[[[-0.4242, -0.42<...>42, -0.4242, -0.4242]]]])
createTheMNISTNet        function      <function createTheMNISTNet at 0x7c9ef9006a20>
datasets                 module        <module 'torchvision.data<...>on/datasets/__init__.py'>
device                   device        cuda
function2trainTheModel   function      <function function2trainT<...>eModel at 0x7c9ef9007920>
loss                     Tensor        tensor(2.3153, grad_fn=<NllLossBackward0>)
lossfun                  NLLLoss       NLLLoss()
net                      mnistNet      mnistNet(\n  (input): Lin<...>eatures=10, bias=True)\n)
nn                       module        <module 'torch.nn' from '<...>es/torch/

In [22]:
import torch

print(f"¿GPU disponible?: {torch.cuda.is_available()}")
print(f"Nombre de la GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'Ninguna'}")

¿GPU disponible?: True
Nombre de la GPU: AMD Radeon RX 6600


In [23]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Entrenando en: {device}")

Entrenando en: cuda


In [24]:
def createTheMNISTNet():

    class mnistNet(nn.Module):
        def __init__(self):
            super().__init__()

            # input layer
            self.input = nn.Linear(784,64) # 784 pixels

            # hidden layer
            self.fc1 = nn.Linear(64,32)
            self.fc2 = nn.Linear(32,32)

            # output layer
            self.output = nn.Linear(32,10)

        def forward(self,x):
            x = x.view(x.shape[0], -1)
            
            x = F.relu( self.input(x) )
            x = F.relu( self.fc1(x) )
            x = F.relu( self.fc2(x) )
            return torch.log_softmax( self.output(x), dim=1 )

    net = mnistNet()

    lossfun = nn.NLLLoss()

    optimizer = torch.optim.SGD(net.parameters(), lr=0.1) # try with Adam later

    return net, lossfun, optimizer

In [25]:
net, lossfun, optimizer = createTheMNISTNet()

X,y = next(iter(train_loader))
yHat = net(X)

print(yHat)

loss = lossfun(yHat, y)
print(' ')
print('Loss:')
print(loss)

tensor([[-2.4826, -2.3720, -2.1282, -2.4890, -2.2554, -2.2382, -2.3982, -2.2580,
         -2.2814, -2.1887],
        [-2.4870, -2.3629, -2.1261, -2.5022, -2.2659, -2.2173, -2.4105, -2.2726,
         -2.2565, -2.1954],
        [-2.5024, -2.3446, -2.1574, -2.4366, -2.2512, -2.1658, -2.4400, -2.3515,
         -2.2635, -2.1819],
        [-2.4518, -2.3670, -2.1333, -2.4517, -2.2788, -2.2073, -2.4366, -2.3312,
         -2.2416, -2.1876],
        [-2.4951, -2.3545, -2.1481, -2.4527, -2.2706, -2.1644, -2.4288, -2.3623,
         -2.2275, -2.1930],
        [-2.5043, -2.3783, -2.1926, -2.4233, -2.2674, -2.2129, -2.3684, -2.3581,
         -2.2270, -2.1519],
        [-2.4979, -2.3799, -2.1670, -2.3932, -2.2705, -2.2017, -2.3570, -2.3700,
         -2.2256, -2.2140],
        [-2.4900, -2.3514, -2.1459, -2.4454, -2.2603, -2.2151, -2.4171, -2.2932,
         -2.2809, -2.1854],
        [-2.4912, -2.3424, -2.1520, -2.4156, -2.2701, -2.1819, -2.4191, -2.3456,
         -2.2549, -2.2098],
        [-2.4985, -

## Training

In [26]:
def function2trainTheModel():

    numepochs = 60

    net, loss, optimizer = createTheMNISTNet()
    net.to(device)

    losses = torch.zeros(numepochs)
    trainAcc = []
    testAcc = []

    for epochi in range(numepochs):

        # ===== Train =====
        batchAcc = []
        batchLoss = []
        for X,y in train_loader:

            X, y = X.to(device), y.to(device)
            
            # forward
            yHat = net(X)
            loss = lossfun(yHat, y)

            # back
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            # loss
            batchLoss.append( loss.item() )

            # accuracy
            matches = torch.argmax(yHat, axis=1) == y # predictions
            matchesNumeric = matches.float() # to numeric
            accuracyPct = 100*torch.mean(matchesNumeric) # pct of mean
            batchAcc.append( accuracyPct ) # add

        trainAcc.append( np.mean(batchAcc) )

        losses[epochi] = np.mean(batchLoss)

        # ===== Test =====
        X,y = next(iter(test_loader))
        X, y = X.to(device), y.to(device)
        yHat = net(X)

        testAcc.append( 100*torch.mean( (torch.argmax(yHat, axis=1)==y).float() ) )

        print(f'Epoch: {epochi} \f trainAcc: {trainAcc[epochi]} \f testAcc: {testAcc[epochi]}')

    return trainAcc, testAcc, losses, net

In [27]:
trainAcc, testAcc, losses, net = function2trainTheModel()

RuntimeError: HIP error: invalid device function
HIP kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing AMD_SERIALIZE_KERNEL=3
Compile with `TORCH_USE_HIP_DSA` to enable device-side assertions.


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16,5))

ax[0].plot(losses)
ax[0].set_xlabel('Epochs')
ax[0].set_ylabel('Loss')
ax[0].set_ylim([0,3])
ax[0].set_title('Model loss')

ax[1].plot(trainAcc, label='Train')
ax[1].plot(testAcc, label='Test')
ax[1].set_xlabel('Epochs')
ax[1].set_ylabel('Accuracy (%)')
ax[1].set_ylim([10,100])
ax[1].set_title(f'Final model test accuracy: {testAcc[-1]:.2f}%')
ax[1].legend()